In [1]:
from datasets import load_dataset
dataset = load_dataset("roneneldan/TinyStories")

print(dataset)
print(dataset['train'].features)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
{'text': Value('string')}


In [2]:
small_train = dataset['train'].select(range(10_000))
print(type(small_train))
small_eval = dataset['validation'].select(range(1_000))

all_text = ''.join(small_train['text']) + ''.join(small_eval['text'])

vocab = sorted(set(all_text))
print(vocab)
print(len(vocab))

print(small_train)

<class 'datasets.arrow_dataset.Dataset'>
['\n', ' ', '!', '"', '$', '&', "'", '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '¡', '¦', '©', '«', '±', '³', '»', 'Â', 'Ã', 'â', 'œ', 'Š', '˜', '“', '”', '€', '™']
94
Dataset({
    features: ['text'],
    num_rows: 10000
})


In [3]:
stoi = {ch:i for i,ch in enumerate(vocab)}
itos = {i:ch for i,ch in enumerate(vocab)}

# Simple Bijection Encoding, BPE not needed for such a small model
encode = lambda seq: [stoi[c] for c in seq]
decode = lambda tok: ''.join([itos[i] for i in tok])

def tokenize(example):
    return {'ids': encode(example['text'])}

tokenized_train = small_train.map(tokenize, remove_columns=['text'])
tokenized_eval   = small_eval.map(tokenize, remove_columns=['text'])
print(type(tokenized_train))


import numpy as np

#Flatten to 1-D arrays for torch
train_ids = np.array([id for row in tokenized_train['ids'] for id in row], dtype=np.uint16)
eval_ids   = np.array([id for row in tokenized_eval['ids']   for id in row], dtype=np.uint16)

<class 'datasets.arrow_dataset.Dataset'>


In [4]:
import torch

train = torch.tensor(train_ids,dtype=torch.long)
eval = torch.tensor(eval_ids,dtype=torch.long)
print(train.shape,train.dtype)
print(train[:100])


torch.Size([8674761]) torch.int64
tensor([39, 64, 55,  1, 54, 51, 75,  8,  1, 51,  1, 62, 59, 70, 70, 62, 55,  1,
        57, 59, 68, 62,  1, 64, 51, 63, 55, 54,  1, 36, 59, 62, 75,  1, 56, 65,
        71, 64, 54,  1, 51,  1, 64, 55, 55, 54, 62, 55,  1, 59, 64,  1, 58, 55,
        68,  1, 68, 65, 65, 63, 10,  1, 43, 58, 55,  1, 61, 64, 55, 73,  1, 59,
        70,  1, 73, 51, 69,  1, 54, 59, 56, 56, 59, 53, 71, 62, 70,  1, 70, 65,
         1, 66, 62, 51, 75,  1, 73, 59, 70, 58])


In [5]:
block_size = 8

x = train[:block_size+1]
y = train[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"Context: {context}, Target:{target}")
   
   

Context: tensor([39]), Target:64
Context: tensor([39, 64]), Target:55
Context: tensor([39, 64, 55]), Target:1
Context: tensor([39, 64, 55,  1]), Target:54
Context: tensor([39, 64, 55,  1, 54]), Target:51
Context: tensor([39, 64, 55,  1, 54, 51]), Target:75
Context: tensor([39, 64, 55,  1, 54, 51, 75]), Target:8
Context: tensor([39, 64, 55,  1, 54, 51, 75,  8]), Target:1


In [6]:
torch.manual_seed(1337)
batch_size = 4 #How many independent sequences do we porcess in parallel
block_size = 8 #Maximum amount of context for transformer

def get_batch(split):
    data = train if split == 'train' else eval

    ix = torch.randint(len(data) -block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x,y

xb, yb = get_batch('train')
print(xb.shape,xb)
print(yb.shape,yb)

torch.Size([4, 8]) tensor([[70, 58, 55,  1, 54, 65, 65, 68],
        [62, 54, 10,  1, 36, 71, 53, 75],
        [ 1, 63, 65, 63,  1, 53, 51, 63],
        [65,  1, 53, 68, 75, 10,  1, 43]])
torch.Size([4, 8]) tensor([[58, 55,  1, 54, 65, 65, 68, 10],
        [54, 10,  1, 36, 71, 53, 75,  1],
        [63, 65, 63,  1, 53, 51, 63, 55],
        [ 1, 53, 68, 75, 10,  1, 43, 58]])


In [7]:
import torch.nn as nn
from torch.nn import functional as F

class Model(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
    
    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx) # Logits(Batch = 4, Time = 8, Channels = len(vocab))

        if targets is None: 
            loss = None
            return logits,loss
        else:

            B,T,C = logits.shape

            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits,loss

    def generate(self,idx,max_new_tokens):

        for _ in range(max_new_tokens):

            logits, _ = self(idx)

            logits = logits[:, -1, :]
            probs = F.softmax(logits,dim=-1)

            idx_next = torch.multinomial(probs,num_samples=1)

            idx = torch.cat((idx,idx_next),dim=1)

        return idx

m = Model(vocab_size=len(vocab))
out,loss = m(xb,yb)
print(out.shape,out)
print(loss)

idx = torch.zeros((1,1),dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist())) 

torch.Size([32, 94]) tensor([[ 0.0412, -1.2073, -1.2257,  ...,  0.7342, -1.4538, -0.4332],
        [ 0.7178,  0.3641,  0.0168,  ...,  0.0134, -1.3952, -1.0559],
        [ 0.6092,  2.2076, -1.1000,  ...,  1.7355, -0.7786,  0.6123],
        ...,
        [ 0.5152,  0.6091,  0.1222,  ..., -0.7856, -0.5349,  0.5812],
        [ 0.1298, -1.9446,  0.0315,  ...,  2.1382,  0.5114,  1.2191],
        [-0.3449,  1.3128, -0.1518,  ..., -0.1210, -0.9201, -0.4817]],
       grad_fn=<ViewBackward0>)
tensor(5.0010, grad_fn=<NllLossBackward0>)

"!«$&œhXqR7rkYOY8Z /;ZÂB-VÂ;œlg€««"!±j-BBRcAry51v9Yu?¡do4&$Še“Ã,Qll
F˜/D€1Uprn2œ¡.3
âa&xYHÃP8Lbc3tq™


In [8]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)

batch_size = 32
for steps in range(30000):

    xb,yb = get_batch('train')

    logits,loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.272413969039917


In [10]:
print(decode(m.generate(idx, max_new_tokens=300)[0].tolist())) 


"


Lik, thean?"Suay, gine Thofor silbadammiress as alove resa vesar. angr m, withendy aidrd foke ou! beretunnnd whe to tcoupave thenthey son afed aloppe he hed s lld.Ond, m s mererndavor str lie uploot il athe gsthe ind ggeck ano hr cle sas a wased iny as sanerey, aig aireseid thessherdoke s ba The
